# 01 Data Preparation

Loads all raw Excel data, cleans/reshapes, and saves **3 Parquet files** to `/MWAP/data/`:

| Output | Contents | Approx Size |
|---|---|---|
| `uncertainty_params.parquet` | 100 rows × 5 cols — case ID + 4 uncertainty parameters | tiny |
| `production_timeseries.parquet` | ~1.1M rows — all 100 cases × 6 metrics × ~11K daily timesteps (long format) | ~50 MB |
| `grid_properties.parquet` | ~4.25M rows — 42,512 active cells × 100 cases × ~20 useful columns | ~500 MB |

**Data quality actions:**
- Filter sentinel values (`-999.25` → inactive cells removed)
- Fix column name typo (` Connateg  Gas Saturation`)
- Drop constant columns (NTG, SWU, SGL, SGAS, RS, RV)
- Parse 2-row headers in production time series
- Normalize all dates to `datetime64`
- Add `case_id` and `split` (train/val/test) columns

In [0]:
%pip install openpyxl fastexcel -q

In [0]:
import pandas as pd
import numpy as np
import os
import time

# Paths
from pathlib import Path
_curr = Path.cwd().resolve()
REPO_ROOT = None
temp = _curr
while temp != temp.parent:
    if (temp / "TestCases").exists():
        REPO_ROOT = temp
        break
    temp = temp.parent

if REPO_ROOT is not None and (REPO_ROOT / "TestCases").exists():
    ROOT = str(REPO_ROOT / "TestCases")
    DATA_DIR = str(REPO_ROOT / "data" / "shared")
else:
    ROOT = "/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP"
    DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Constants
SENTINEL = -999.25

# Case split mapping
def get_split(case_num: int) -> str:
    if case_num <= 70:
        return "train"
    elif case_num <= 85:
        return "validation"
    else:
        return "test"

# Columns to drop (constant across all active cells in all cases)
CONSTANT_COLS = [
    "Net-to-Gross Ratio (NTG)",       # always 1.0
    "Maximum Water Saturation (SWU)",  # always 1.0
    " Connateg  Gas Saturation (SGL)", # always 0.0 (also has typo)
    "Gas Saturation (SGAS)",           # always 0.0
    "Solution Gas-Oil Ratio (RS)",     # always 0.3633
    "Vapor Oil-Gas Ratio (RV)",        # always 0.0
]

# Column rename map (fix typos)
COL_RENAME = {
    " Connateg  Gas Saturation (SGL)": "Connate Gas Saturation (SGL)",
}

# Grid case file mapping
GRID_FILES = [
    ("03 Train Cases 1.xlsx",   list(range(1, 21))),
    ("04 Train Cases 2.xlsx",   list(range(21, 41))),
    ("05 Train Cases 3.xlsx",   list(range(41, 61))),
    ("06 Train Cases 4.xlsx",   list(range(61, 71))),
    ("07  Validation Cases.xlsx", list(range(71, 86))),
    ("08 Test Cases.xlsx",      list(range(86, 101))),
]

print("Config ready.")
print(f"Data output: {DATA_DIR}")
print(f"Grid files to process: {sum(len(c) for _, c in GRID_FILES)} cases across {len(GRID_FILES)} files")

In [0]:
# ── Load uncertainty parameters ───────────────────────────────
df_unc = pd.read_excel(
    os.path.join(ROOT, "01 Introduction.xlsx"),
    sheet_name="Uncertainty Values",
    engine="openpyxl",
)

# Clean
df_unc = df_unc.rename(columns={"Item": "case_id"})
df_unc["case_num"] = df_unc["case_id"].str.extract(r"(\d+)").astype(int)
df_unc["split"] = df_unc["case_num"].map(get_split)

# Reorder
df_unc = df_unc[["case_num", "case_id", "split",
                  "Fault Transmissibility", "Porosity Multiplier",
                  "Permeability Multiplier", "Aquifer Pore Volume"]]
df_unc = df_unc.sort_values("case_num").reset_index(drop=True)

# Save
out_path = os.path.join(DATA_DIR, "uncertainty_params.parquet")
df_unc.to_parquet(out_path, index=False)

# Summary for Brain
print(f"\u2705 Saved: {out_path}")
print(f"   Shape: {df_unc.shape}")
print(f"   Split counts: {df_unc['split'].value_counts().to_dict()}")
print(f"\n{df_unc.describe().to_string()}")
print(f"\n{df_unc.head(5).to_string()}")

In [0]:
# ── Load all 6 production time series sheets ──────────────────
PROD_FILE = os.path.join(ROOT, "02 Field Production.xlsx")

SHEETS = [
    ("Oil production cumulative",   "oil_cum",   "STB"),
    ("Oil production rate",         "oil_rate",  "STB/d"),
    ("Gas production cumulative",   "gas_cum",   "MSCF"),
    ("Gas production rate",         "gas_rate",  "MSCF/d"),
    ("Water production cumulative", "water_cum", "STB"),
    ("Water production rate",       "water_rate", "STB/d"),
]

all_dfs = []

for sheet_name, metric_key, unit in SHEETS:
    print(f"  Loading: {sheet_name}...")
    raw = pd.read_excel(PROD_FILE, sheet_name=sheet_name, engine="openpyxl")
    
    # Row 0 = case names, Row 1 = metric labels, data from Row 2
    case_names = raw.iloc[0, 1:].values  # Case_1 ... Case_100, RAMP_History
    data = raw.iloc[2:].copy()
    data.columns = ["date"] + list(case_names)
    data["date"] = pd.to_datetime(data["date"])
    
    # Melt to long format
    melted = data.melt(id_vars="date", var_name="case_raw", value_name="value")
    melted["value"] = pd.to_numeric(melted["value"], errors="coerce")
    melted["metric"] = metric_key
    melted["unit"] = unit
    
    # Parse case number
    melted["case_num"] = melted["case_raw"].str.extract(r"(\d+)").astype(float)
    melted.loc[melted["case_raw"] == "RAMP_History", "case_num"] = 0  # 0 = observed history
    melted["case_num"] = melted["case_num"].astype(int)
    melted["split"] = melted["case_num"].map(lambda x: "history" if x == 0 else get_split(x))
    
    melted = melted.drop(columns=["case_raw"])
    all_dfs.append(melted)

df_prod = pd.concat(all_dfs, ignore_index=True)

# Also add the quarterly field production (observed history rates)
df_fp = pd.read_excel(PROD_FILE, sheet_name="Field Production", engine="openpyxl")
df_fp["Date"] = pd.to_datetime(df_fp["Date"])
field_rows = []
for _, row in df_fp.iterrows():
    d = row["Date"]
    for old_col, metric_key, unit in [
        ("Oil production rate (STB/d)",   "oil_rate_quarterly",  "STB/d"),
        ("Gas production rate (MSCF/d)",  "gas_rate_quarterly",  "MSCF/d"),
        ("Water production rate (STB/d)", "water_rate_quarterly", "STB/d"),
        ("Water injection rate (STB/d)",  "water_inj_quarterly", "STB/d"),
    ]:
        field_rows.append({
            "date": d, "value": row[old_col], "metric": metric_key,
            "unit": unit, "case_num": 0, "split": "history"
        })

df_field = pd.DataFrame(field_rows)
df_prod = pd.concat([df_prod, df_field], ignore_index=True)

# Sort
df_prod = df_prod.sort_values(["case_num", "metric", "date"]).reset_index(drop=True)

# Save
out_path = os.path.join(DATA_DIR, "production_timeseries.parquet")
df_prod.to_parquet(out_path, index=False)

# Summary
print(f"\n\u2705 Saved: {out_path}")
print(f"   Shape: {df_prod.shape}")
print(f"   Metrics: {df_prod['metric'].unique().tolist()}")
print(f"   Date range: {df_prod['date'].min()} → {df_prod['date'].max()}")
print(f"   Cases: {df_prod['case_num'].nunique()} (0=history, 1-100=sim)")
print(f"   Split counts: {df_prod.groupby('split')['case_num'].nunique().to_dict()}")
print(f"\n   Rows per metric:")
print(df_prod.groupby("metric")["value"].agg(["count", "min", "max", "mean"]).to_string())

In [0]:
# ── Load grid-cell properties for all 100 cases ──────────────
# Strategy: process each Excel file → save per-file parquet → merge at end
# Uses calamine (Rust) engine for ~10x faster Excel reads
# Resumes from already-saved parts if found

import gc

EXCEL_ENGINE = "openpyxl"

cols_to_drop = set(CONSTANT_COLS)
total_start = time.time()
part_files = []

for i, (fname, case_nums) in enumerate(GRID_FILES, 1):
    part_path = os.path.join(DATA_DIR, f"grid_part{i}.parquet")
    
    # Skip if already saved from a previous run
    if os.path.exists(part_path):
        part_files.append(part_path)
        print(f"\n✅ [{i}/{len(GRID_FILES)}] {fname} — already saved, skipping")
        continue
    
    fpath = os.path.join(ROOT, fname)
    print(f"\n📂 [{i}/{len(GRID_FILES)}] {fname} ({len(case_nums)} cases)")
    
    file_parts = []
    for case_num in case_nums:
        t0 = time.time()
        df = pd.read_excel(fpath, sheet_name=f"Case {case_num}", engine=EXCEL_ENGINE)
        df = df.rename(columns=COL_RENAME)
        df = df[df["Porosity (PORO)"] != SENTINEL].copy()
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")
        df["case_num"] = case_num
        df["split"] = get_split(case_num)
        file_parts.append(df)
        print(f"  Case {case_num:3d}: {len(df):,} active cells | {time.time()-t0:.1f}s")
    
    pd.concat(file_parts, ignore_index=True).to_parquet(part_path, index=False)
    part_files.append(part_path)
    del file_parts
    gc.collect()
    print(f"  ✅ Saved part {i}")

print(f"\n⏱ Excel loading: {(time.time() - total_start)/60:.1f} min")

# Merge all parts
print("\nMerging parts...")
df_grid = pd.concat([pd.read_parquet(p) for p in part_files], ignore_index=True)

# Verify PERMX == PERMY
is_isotropic = np.allclose(
    df_grid["Permeability I (PERMX)"].values,
    df_grid["Permeability J (PERMY)"].values
)
print(f"PERMX == PERMY across all cases: {is_isotropic}")
if is_isotropic:
    df_grid = df_grid.drop(columns=["Permeability J (PERMY)"])
    print("  → Dropped PERMY (redundant)")

# Save final
out_path = os.path.join(DATA_DIR, "grid_properties.parquet")
df_grid.to_parquet(out_path, index=False)

# Clean up part files
for p in part_files:
    os.remove(p)

print(f"\n✅ Saved: {out_path}")
print(f"   Shape: {df_grid.shape}")
print(f"   Columns: {list(df_grid.columns)}")
print(f"   Cases: {df_grid['case_num'].nunique()}")
print(f"   Active cells per case: {df_grid.groupby('case_num').size().unique()}")
print(f"   Splits: {df_grid.groupby('split')['case_num'].nunique().to_dict()}")
print(f"\n{df_grid.describe().to_string()}")

In [0]:
# ── Final report ──────────────────────────────────────────────
print("=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)

for fname in ["uncertainty_params.parquet", "production_timeseries.parquet", "grid_properties.parquet"]:
    fpath = os.path.join(DATA_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    df_check = pd.read_parquet(fpath)
    print(f"\n📄 {fname}")
    print(f"   Size: {size_mb:.1f} MB")
    print(f"   Shape: {df_check.shape}")
    print(f"   Columns: {list(df_check.columns)}")
    del df_check

print("\n" + "=" * 60)
print("All downstream notebooks should load from /MWAP/data/*.parquet")
print("=" * 60)